# Manhattans and QQ plots 
1. GP2 AFR
2. AAC meta
3. AFR/AAC meta

In [2]:
## using python 3.10 custom gwaslab env 
## Import the necessary packages 
import os
import numpy as np
import pandas as pd
import gwaslab as gl
import math
import sys
import subprocess
import statsmodels.api as sm
import scipy
from scipy import stats
from scipy.stats import chi2
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

## Print out package versions
## Getting packages loaded into this notebook and their versions to allow for reproducibility
import pkg_resources
import types
from datetime import date

today = date.today()
date = today.strftime("%d-%b-%Y").upper()

## Define function 
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            name = val.__name__.split(".")[0]
        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages:
            name = poorly_named_packages[name]

        yield name

## Get a list of packages imported 
imports = list(set(get_imports()))

requirements = []
for m in pkg_resources.working_set:
    if m.project_name in imports and m.project_name != "pip":
        requirements.append((m.project_name, m.version))

## Print out packages and versions 
print(f"PACKAGE VERSIONS ({date})")
for r in requirements:
    print("\t{}=={}".format(*r))

## Also print which Python is being used
print("\nPYTHON INFO")
print(f"\tPython executable: {sys.executable}")

PACKAGE VERSIONS (22-NOV-2025)
	seaborn==0.13.2
	statsmodels==0.14.4
	gwaslab==3.6.8
	matplotlib==3.8.4
	numpy==1.26.4
	pandas==2.3.2
	scipy==1.15.3

PYTHON INFO
	Python executable: /vf/users/makariousmb/conda/envs/gwaslab_310/bin/python


/tmp/ipykernel_3828436/1682565989.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
# Set your references as needed
gl.options.set_option("config","/data/gwaslab/data/config.json")
gl.options.set_option("reference","/data/gwaslab/data/reference.json")
gl.options.set_option("formatbook","/data/gwaslab/data/formatbook.json")
gl.options.set_option("data_directory","/data/gwaslab/.gwaslab/")

# Manhattans and QQ Plots

## GP2 AFR 

In [ ]:
# Load harmonized summary stats
afr_mungedstats = f"{WORK_DIR}/data/GP2_R11/AFR/GP2_AFR_GWAS_R11.wAlleles.FOR_PLINK.txt"

afr_mungedstats = gl.Sumstats(afr_mungedstats,
                       snpid="SNP",
                       chrom="CHR",
                       pos="BP",
                       ea="A1",
                       nea="A2",
                       beta="BETA",
                       se="SE",
                       p="P",
                       n="NMISS")

In [23]:
afr_mungedstats.get_lead(
           scaled=False,
           use_p=False,
           windowsizekb=500,
           sig_level=5e-8,
           anno=False,
           build="38",
           source="ensembl",
           verbose=True,
           gls=False)

2025/11/17 17:26:00 Start to extract lead variants...v3.6.0
2025/11/17 17:26:00  -Current Dataframe shape : 14662187 x 10 ; Memory usage: 960.59 MB
2025/11/17 17:26:00  -Processing 14662187 variants...
2025/11/17 17:26:00  -Significance threshold : 5e-08
2025/11/17 17:26:00  -Sliding window size: 500  kb
2025/11/17 17:26:06  -Using P for extracting lead variants...
2025/11/17 17:26:06  -Found 264 significant variants in total...
2025/11/17 17:26:06  -Identified 5 lead variants!
2025/11/17 17:26:06 Finished extracting lead variants.


,SNPID,CHR,POS,EA,NEA,BETA,SE,P,N,STATUS
650142,chr1:155235878:G:T,1,155235878,T,G,-0.548387,0.042762,1.201710e-37,6273.823933,9999999
3961636,chr4:89704960:G:A,4,89704960,A,G,-0.284923,0.039329,4.331670e-13,6273.823933,9999999
10426870,chr12:40309145:C:T,12,40309145,T,C,0.546907,0.097002,1.719640e-08,6273.823933,9999999
10608193,chr12:75689052:G:A,12,75689052,A,G,-0.324572,0.059331,4.486460e-08,6273.823933,9999999
12679807,chr16:76969646:G:C,16,76969646,C,G,-0.798481,0.141486,1.665790e-08,6273.823933,9999999


In [ ]:
afr_mungedstats.plot_mqq(sig_level_lead=5e-8,
                      build="38",
                      windowsizekb=500,
                      sig_level=5e-8,
                      anno=True,
                      anno_set=["chr1:155235878:G:T", "chr4:89704960:G:A", "chr12:40309145:C:T", "chr12:75689052:G:A", "chr16:76969646:G:C"],
                      anno_alias=
                         {"chr1:155235878:G:T":"rs3115534", 
                          "chr4:89704960:G:A":"rs356182", 
                          "chr12:40309145:C:T":"rs72546327", 
                          "chr12:75689052:G:A":"rs12302417",
                          "chr16:76969646:G:C":"rs113244182"},
                      highlight=["chr1:155235878:G:T", "chr4:89704960:G:A", "chr12:40309145:C:T", "chr12:75689052:G:A", "chr16:76969646:G:C"], 
                      pinpoint=["chr1:155235878:G:T", "chr4:89704960:G:A", "chr12:40309145:C:T", "chr12:75689052:G:A", "chr16:76969646:G:C"],
                      anno_style="expand",
                      sig_line=True,
                      save_args={"dpi":400,"facecolor":"white"},
                      save=f"{WORK_DIR}/results/plots/GP2-AFR-Manhattan-QQplot-wANNO.png")

## AAC Only Meta: GP2 AAC / 23andMe AAC / MVP AAC

In [ ]:
# Load harmonized summary stats
aac_meta_path = f"{WORK_DIR}/results/AAC_META_GP2_23andMe_MVP/GP2_R11_AAC_23andMe_MVP_GWAS.wAlleles.PLINK_meta.meta"

aac_meta = gl.Sumstats(aac_meta_path, build="38",
                       snpid="SNP",
                       chrom="CHR",
                       pos="BP",
                       ea="A1",
                       nea="A2",
                       beta="BETA",
                       p="P", sep="\s+")

In [6]:
aac_meta.get_lead(
           scaled=False,
           use_p=False,
           windowsizekb=500,
           sig_level=5e-8,
           anno=False,
           build="38",
           source="ensembl",
           verbose=True,
           gls=False)

2025/11/19 12:17:24 Start to extract lead variants...v3.6.0
2025/11/19 12:17:24  -Current Dataframe shape : 14880132 x 8 ; Memory usage: 717.96 MB
2025/11/19 12:17:24  -Processing 14880132 variants...
2025/11/19 12:17:24  -Significance threshold : 5e-08
2025/11/19 12:17:24  -Sliding window size: 500  kb
2025/11/19 12:17:28  -Using P for extracting lead variants...
2025/11/19 12:17:28  -Found 2 significant variants in total...
2025/11/19 12:17:28  -Identified 1 lead variants!
2025/11/19 12:17:28 Finished extracting lead variants.


,SNPID,CHR,POS,EA,NEA,BETA,P,STATUS
650834,chr1:155235878:G:T,1,155235878,T,G,-0.3317,1.357000e-09,3899999


In [ ]:
aac_meta.plot_mqq(sig_level_lead=5e-8,
                      build="38",
                      windowsizekb=500,
                      sig_level=5e-8,
                      anno=True,
                      anno_set=["chr1:155235878:G:T"],
                      anno_alias=
                         {"chr1:155235878:G:T":"rs3115534"}, 
                      highlight=["chr1:155235878:G:T"],
                      anno_style="right",
                      sig_line=True,
                      save_args={"dpi":400,"facecolor":"white"},
                      save=f"{WORK_DIR}/results/plots/AAC-ONLY-META-GP2-23andMe-MVP-Manhattan-QQplot-wANNO.png")

# AFR/AAC Meta-analysis: GP2 AAC / GP2 AFR / 23andMe AAC / MVP AAC

In [4]:
# Load harmonized summary stats
gp2_23andMe_MVP_path = f"{WORK_DIR}/results/META_GP2_23andMe_MVP/GP2_R11_AAC_AFR_23andMe_MVP_GWAS.wAlleles.PLINK_meta.meta"

gp2_23andMe_MVP = gl.Sumstats(gp2_23andMe_MVP_path, build="38",
                       snpid="SNP",
                       chrom="CHR",
                       pos="BP",
                       ea="A1",
                       nea="A2",
                       beta="BETA",
                       p="P", sep="\s+")

2025/11/22 13:50:39 GWASLab v3.6.0 https://cloufield.github.io/gwaslab/
2025/11/22 13:50:39 (C) 2022-2025, Yunye He, Kamatani Lab, GPL-3.0 license, gwaslab@gmail.com
2025/11/22 13:50:39 Python version: 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:45:41) [GCC 13.3.0]
2025/11/22 13:50:39 Start to initialize gl.Sumstats from file :/data/CARD_AA/projects/2025_2026_AFR_AAC_GWAS/results/META_GP2_23andMe_MVP/GP2_R11_AAC_AFR_23andMe_MVP_GWAS.wAlleles.PLINK_meta.meta
2025/11/22 13:51:02  -Reading columns          : A2,A1,BETA,CHR,BP,P,SNP
2025/11/22 13:51:02  -Renaming columns to      : NEA,EA,BETA,CHR,POS,P,SNPID
2025/11/22 13:51:02  -Current Dataframe shape : 15685296  x  7
2025/11/22 13:51:02  -Initiating a status column: STATUS ...
2025/11/22 13:51:02  -Genomic coordinates are based on GRCh38/hg38...
2025/11/22 13:51:05 Start to reorder the columns...v3.6.0
2025/11/22 13:51:05  -Current Dataframe shape : 15685296 x 8 ; Memory usage: 771.63 MB
2025/11/22 13:51:05  -Reordering c

In [5]:
gp2_23andMe_MVP.get_lead(
           scaled=False,
           use_p=False,
           windowsizekb=500,
           sig_level=5e-8,
           anno=False,
           build="38",
           source="ensembl",
           verbose=True,
           gls=False)

2025/11/22 13:51:06 Start to extract lead variants...v3.6.0
2025/11/22 13:51:06  -Current Dataframe shape : 15685296 x 8 ; Memory usage: 786.59 MB
2025/11/22 13:51:06  -Processing 15685296 variants...
2025/11/22 13:51:06  -Significance threshold : 5e-08
2025/11/22 13:51:06  -Sliding window size: 500  kb
2025/11/22 13:51:09  -Using P for extracting lead variants...
2025/11/22 13:51:09  -Found 316 significant variants in total...
2025/11/22 13:51:09  -Identified 4 lead variants!
2025/11/22 13:51:09 Finished extracting lead variants.


,SNPID,CHR,POS,EA,NEA,BETA,P,STATUS
689660,chr1:155235878:G:T,1,155235878,T,G,-0.4662,1.533000e-43,3899999
4138757,chr4:76213633:C:T,4,76213633,T,C,0.1822,3.505000e-08,3899999
4215701,chr4:89704960:G:A,4,89704960,A,G,-0.2373,8.375000e-16,3899999
11139099,chr12:40027276:T:C,12,40027276,C,T,0.6251,1.252000e-08,3899999


In [ ]:
gp2_23andMe_MVP.plot_mqq(sig_level_lead=5e-8,
                      build="38",
                      windowsizekb=500,
                      sig_level=5e-8,
                      anno=True,
                      anno_set=["chr1:155235878:G:T", "chr4:76213633:C:T", "chr4:89704960:G:A", "chr12:40027276:T:C"],
                      anno_alias=
                         {"chr1:155235878:G:T":"rs3115534", 
                          "chr4:76213633:C:T":"rs11547135", 
                          "chr4:89704960:G:A":"rs356182", 
                          "chr12:40027276:T:C":"rs139283662"}, 
                      highlight=["chr1:155235878:G:T", "chr4:76213633:C:T", "chr4:89704960:G:A", "chr12:40027276:T:C"],
                      anno_style="right",
                      sig_line=True,
                      save_args={"dpi":400,"facecolor":"white"},
                      save=f"{WORK_DIR}/results/plots/AFR-AAC-META-GP2-23andMe-MVP-Manhattan-QQplot-wANNO.png")

In [17]:
! head -1 {WORK_DIR}/results/META_GP2_23andMe_MVP/GP2_R11_AAC_AFR_23andMe_MVP_GWAS.wAlleles.PLINK_meta.meta | cut -f 1-6
! grep "chr12:40309145:C:T" {WORK_DIR}/results/META_GP2_23andMe_MVP/GP2_R11_AAC_AFR_23andMe_MVP_GWAS.wAlleles.PLINK_meta.meta | cut -f 1-6

 CHR          BP            SNP  A1  A2   N           P        P(R)    BETA BETA(R)       Q       I  WEIGHTED_Z       P(WZ)      F0      F1      F2      F3
  12    40309145 chr12:40309145:C:T   T   C   4   6.506e-07      0.1468  0.4000  0.2485  0.0415   63.54       4.493   7.014e-06 -0.1593  0.5469  0.2643  0.0745


# GP2 Analyses 

## GP2 AAC

In [4]:
# Load harmonized summary stats
aac_mungedstats = f"{WORK_DIR}/data/GP2_R11/AAC/GP2_AAC_GWAS_R11.wAlleles.FOR_PLINK.txt"

aac_sumstats = gl.Sumstats(aac_mungedstats,
                       snpid="SNP",
                       chrom="CHR",
                       pos="BP",
                       ea="A1",
                       nea="A2",
                       beta="BETA",
                       se="SE",
                       p="P",
                       n="NMISS")

2025/11/17 16:02:19 GWASLab v3.6.0 https://cloufield.github.io/gwaslab/
2025/11/17 16:02:19 (C) 2022-2025, Yunye He, Kamatani Lab, GPL-3.0 license, gwaslab@gmail.com
2025/11/17 16:02:19 Python version: 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:45:41) [GCC 13.3.0]
2025/11/17 16:02:19 Start to initialize gl.Sumstats from file :/data/CARD_AA/projects/2025_2026_AFR_AAC_GWAS/data/GP2_R11/AAC/GP2_AAC_GWAS_R11.wAlleles.FOR_PLINK.txt
2025/11/17 16:02:41  -Reading columns          : A1,NMISS,CHR,BP,BETA,SNP,SE,P,A2
2025/11/17 16:02:41  -Renaming columns to      : EA,N,CHR,POS,BETA,SNPID,SE,P,NEA
2025/11/17 16:02:41  -Current Dataframe shape : 14411150  x  9
2025/11/17 16:02:42  -Initiating a status column: STATUS ...
2025/11/17 16:02:42  #WARNING! Version of genomic coordinates is unknown...
2025/11/17 16:02:47 Start to reorder the columns...v3.6.0
2025/11/17 16:02:47  -Current Dataframe shape : 14411150 x 10 ; Memory usage: 930.79 MB
2025/11/17 16:02:47  -Reordering columns to

In [5]:
aac_sumstats.get_lead(
           scaled=False,
           use_p=False,
           windowsizekb=500,
           sig_level=5e-8,
           anno=False,
           build="38",
           source="ensembl",
           verbose=True,
           gls=False)

2025/11/17 16:04:12 Start to extract lead variants...v3.6.0
2025/11/17 16:04:12  -Current Dataframe shape : 14411150 x 10 ; Memory usage: 944.54 MB
2025/11/17 16:04:12  -Processing 14411150 variants...
2025/11/17 16:04:12  -Significance threshold : 5e-08
2025/11/17 16:04:12  -Sliding window size: 500  kb
2025/11/17 16:04:18  -Using P for extracting lead variants...
2025/11/17 16:04:18  -Found 13 significant variants in total...
2025/11/17 16:04:18  -Identified 1 lead variants!
2025/11/17 16:04:18 Finished extracting lead variants.


,SNPID,CHR,POS,EA,NEA,BETA,SE,P,N,STATUS
637646,chr1:155235878:G:T,1,155235878,T,G,-0.771908,0.128187,1.725620e-09,1191.255668,9999999


In [ ]:
aac_sumstats.plot_mqq(sig_level_lead=5e-8,
                      build="38",
                      windowsizekb=500,
                      sig_level=5e-8,
                      anno=True,
                      anno_set=["chr1:155235878:G:T"],
                      anno_alias={"chr1:155235878:G:T":"rs3115534"},
                      highlight=["chr1:155235878:G:T"], 
                      pinpoint=["chr1:155235878:G:T"],
                      anno_style="expand",
                      sig_line=True,
                      save_args={"dpi":400,"facecolor":"white"},
                      save=f"{WORK_DIR}/results/plots/GP2-AAC-Manhattan-QQplot-wANNO.png")

## GP2 AFR/AAC Meta-analysis

In [ ]:
# Load harmonized summary stats
gp2_mungedstats = f"{WORK_DIR}/results/GP2_R11/GP2_AAC_AFR_GWAS_R11.wAlleles.PLINK_meta.meta"

gp2_mungedstats = gl.Sumstats(gp2_mungedstats,
                       snpid="SNP",
                       chrom="CHR",
                       pos="BP",
                       ea="A1",
                       nea="A2",
                       beta="BETA",
                       p="P", sep="\s+")

In [27]:
gp2_mungedstats.get_lead(
           scaled=False,
           use_p=False,
           windowsizekb=500,
           sig_level=5e-8,
           anno=False,
           build="38",
           source="ensembl",
           verbose=True,
           gls=False)

2025/11/17 17:55:23 Start to extract lead variants...v3.6.0
2025/11/17 17:55:23  -Current Dataframe shape : 13437485 x 8 ; Memory usage: 677.26 MB
2025/11/17 17:55:23  -Processing 13437485 variants...
2025/11/17 17:55:23  -Significance threshold : 5e-08
2025/11/17 17:55:23  -Sliding window size: 500  kb
2025/11/17 17:55:28  -Using P for extracting lead variants...
2025/11/17 17:55:28  -Found 341 significant variants in total...
2025/11/17 17:55:29  -Identified 4 lead variants!
2025/11/17 17:55:29 Finished extracting lead variants.


,SNPID,CHR,POS,EA,NEA,BETA,P,STATUS
594915,chr1:155235878:G:T,1,155235878,T,G,-0.5708,5.750000e-45,9999999
615196,chr1:159204893:T:C,1,159204893,C,T,0.4611,2.243000e-08,9999999
3555539,chr4:76213633:C:T,4,76213633,T,C,0.2307,2.181000e-08,9999999
3623666,chr4:89704960:G:A,4,89704960,A,G,-0.2668,1.964000e-13,9999999


In [ ]:
gp2_mungedstats.plot_mqq(sig_level_lead=5e-8,
                      build="38",
                      windowsizekb=500,
                      sig_level=5e-8,
                      anno=True,
                      anno_set=["chr1:155235878:G:T", "chr1:159204893:T:C", "chr4:76213633:C:T", "chr4:89704960:G:A"],
                      anno_alias=
                         {"chr1:155235878:G:T":"rs3115534", 
                          "chr1:159204893:T:C":"rs2814778", 
                          "chr4:76213633:C:T":"rs11547135", 
                          "chr4:89704960:G:A":"rs356182"},
                      highlight=["chr1:155235878:G:T", "chr1:159204893:T:C", "chr4:76213633:C:T", "chr4:89704960:G:A"], 
                      pinpoint=["chr1:155235878:G:T", "chr1:159204893:T:C", "chr4:76213633:C:T", "chr4:89704960:G:A"],
                      anno_style="right",
                      sig_line=True,
                      save_args={"dpi":400,"facecolor":"white"},
                      save=f"{WORK_DIR}/results/plots/GP2-META-Manhattan-QQplot-wANNO.png")

### Plot with AFR hits 

In [ ]:
gp2_mungedstats.plot_mqq(sig_level_lead=5e-8,
                      build="38",
                      windowsizekb=500,
                      sig_level=5e-8,
                      anno=True,
                      anno_set=["chr1:155235878:G:T", "chr4:89704960:G:A", "chr12:40309145:C:T", "chr12:75689052:G:A", "chr16:76969646:G:C"],
                      anno_alias=
                         {"chr1:155235878:G:T":"rs3115534", 
                          "chr4:89704960:G:A":"rs356182", 
                          "chr12:40309145:C:T":"rs72546327", 
                          "chr12:75689052:G:A":"rs12302417",
                          "chr16:76969646:G:C":"rs113244182"},
                      highlight=["chr1:155235878:G:T", "chr4:89704960:G:A", "chr12:40309145:C:T", "chr12:75689052:G:A", "chr16:76969646:G:C"], 
                      pinpoint=["chr1:155235878:G:T", "chr4:89704960:G:A", "chr12:40309145:C:T", "chr12:75689052:G:A", "chr16:76969646:G:C"],
                      anno_style="expand",
                      sig_line=True,
                      save_args={"dpi":400,"facecolor":"white"},
                      save=f"{WORK_DIR}/results/plots/GP2-META-Manhattan-QQplot-wANNO-wAFRHITS.png")